# 09 — Generalizability Check: AMES Mutagenicity (~7,255 molecules)

**Scope, stated explicitly:** this is a single-seed run of the 5 core models (1D, 2D, 3D
weighted_mean, multimodal concat, multimodal gated) on a second, ~11x larger TDC endpoint --
not a full replication of the hERG protocol. No multi-seed averaging, no aggregation-mode
ablation (uniform_mean/learned_attention), no leave-one-out fusion grid. The purpose is a
generalizability spot-check: does the same architecture, same hyperparameters, same code,
produce sensible results on a differently-sized, differently-behaved endpoint -- not a second
full study.

All hyperparameters are reused unchanged from `config/model_1d.yaml`, `model_2d.yaml`,
`model_3d.yaml`, `model_fusion.yaml` -- no re-tuning for this dataset, so any performance
difference reflects the dataset, not a tuning advantage.


In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

import time
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
import torch
from torch.utils.data import DataLoader
from torch_geometric.loader import DataLoader as PyGDataLoader
from rdkit import Chem
from rdkit import RDLogger
from tqdm.auto import tqdm
from tdc.benchmark_group import admet_group

from src.utils import load_json, save_json, set_seed
from src.featurizers import smiles_to_selfies, build_vocab, PAD
from src.conformers import generate_conformers
from src.graph_featurizer import mol_to_graph_data, ATOM_FEAT_DIM, BOND_FEAT_DIM
from src.datasets import (
    SelfiesDataset, ConformerDataset, collate_conformers, MultimodalDataset, multimodal_collate,
)
from src.models import Selfies1DModel, Graph2DModel, Conformer3DModel, MultimodalModel
from src.train_utils import train_binary_classifier, evaluate

RDLogger.DisableLog("rdApp.*")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEV_SEED = 1
DATASET_NAME = "AMES"

with open("../config/model_1d.yaml") as f: cfg_1d = yaml.safe_load(f)
with open("../config/model_2d.yaml") as f: cfg_2d = yaml.safe_load(f)
with open("../config/model_3d.yaml") as f: cfg_3d = yaml.safe_load(f)
with open("../config/model_fusion.yaml") as f: cfg_fusion = yaml.safe_load(f)

# AMES-specific paths -- kept fully separate from hERG's, nothing collides
AMES_RAW = "../data/raw_ames"
AMES_CONFORMERS_DIR = "../data/processed/conformers_ames"
AMES_SPLITS_FILE = "../data/splits_ames.json"
AMES_VOCAB_FILE = "../data/processed/selfies_vocab_ames.json"
print("device:", device)


c:\Users\ajaya\anaconda3\envs\chemprop\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cuda


## Load AMES via the official TDC benchmark group (same protocol as notebook 01)

In [2]:
Path(AMES_RAW).mkdir(parents=True, exist_ok=True)

group = admet_group(path=AMES_RAW)
benchmark = group.get(DATASET_NAME)
train, valid = group.get_train_valid_split(benchmark=DATASET_NAME, split_type="default", seed=DEV_SEED)
test = benchmark["test"]

print(f"train={len(train)}  valid={len(valid)}  test={len(test)}  (total={len(train)+len(valid)+len(test)})")


Found local copy...
generating training, validation splits...
100%|██████████| 5821/5821 [00:01<00:00, 3735.28it/s]

train=5093  valid=728  test=1457  (total=7278)


## Quick sanity checks (compressed version of notebook 01's EDA)

In [3]:
for name, df in [("train", train), ("valid", valid), ("test", test)]:
    print(f"{name:6s}  n={len(df):5d}  positive_rate={df['Y'].mean():.3f}")

def parseable(df, col="Drug"):
    return df[col].apply(lambda s: Chem.MolFromSmiles(s) is not None).mean()

for name, df in [("train", train), ("valid", valid), ("test", test)]:
    print(f"{name:6s}  RDKit-parseable: {parseable(df):.4f}")


train   n= 5093  positive_rate=0.521
valid   n=  728  positive_rate=0.618
test    n= 1457  positive_rate=0.597
train   RDKit-parseable: 1.0000
valid   RDKit-parseable: 1.0000
test    RDKit-parseable: 1.0000


## Conformer generation -- timing estimate first

At ~11x hERG's molecule count, this is the step most likely to take real wall-clock time. Estimating before committing to the full pass.

In [4]:
all_smiles = pd.concat([train, valid, test])["Drug"].tolist()
all_ids = pd.concat([train, valid, test])["Drug_ID"].tolist()
print(f"Total molecules needing conformers: {len(all_smiles)}")

sample_n = 20
t0 = time.time()
for smi in all_smiles[:sample_n]:
    generate_conformers(smi, **cfg_3d.get("conformers", {
        "n_confs": 8, "prune_rms_thresh": 0.5, "max_iters": 500,
        "forcefield": "MMFF94", "energy_window_kcal": 10.0, "random_seed": 42,
    }))
elapsed = time.time() - t0
per_mol = elapsed / sample_n
est_total_min = per_mol * len(all_smiles) / 60
print(f"~{per_mol:.3f} sec/molecule -> estimated total: {est_total_min:.1f} minutes for {len(all_smiles)} molecules")


Total molecules needing conformers: 7278
~0.074 sec/molecule -> estimated total: 9.0 minutes for 7278 molecules


In [5]:
conformer_cfg = {
    "n_confs": 8, "prune_rms_thresh": 0.5, "max_iters": 500,
    "forcefield": "MMFF94", "energy_window_kcal": 10.0, "random_seed": 42,
}

out_dir = Path(AMES_CONFORMERS_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

manifest = {"success": [], "failed": []}
t_start = time.time()
for mol_id, smi in tqdm(list(zip(all_ids, all_smiles)), total=len(all_ids)):
    out_path = out_dir / f"{mol_id}.npz"
    if out_path.exists():
        manifest["success"].append(mol_id)
        continue
    result = generate_conformers(smi, **conformer_cfg)
    if result is None:
        manifest["failed"].append({"id": mol_id, "smiles": smi})
        continue
    np.savez_compressed(
        out_path, atomic_nums=result["atomic_nums"], coords=result["coords"],
        energies_kcal=result["energies_kcal"], boltzmann_weights=result["boltzmann_weights"],
    )
    manifest["success"].append(mol_id)

elapsed_min = (time.time() - t_start) / 60
print(f"Done in {elapsed_min:.1f} min. Success: {len(manifest['success'])}  Failed: {len(manifest['failed'])}")
save_json(manifest, "../data/processed/ames_conformer_manifest.json")


100%|██████████| 7278/7278 [01:14<00:00, 98.28it/s]   

Done in 1.2 min. Success: 7275  Failed: 3


## Drop failed molecules from all splits

In [6]:
dropped_ids = {f["id"] for f in manifest["failed"]}
print(f"Dropping {len(dropped_ids)} molecules that failed conformer generation.")

train_f = train[~train["Drug_ID"].isin(dropped_ids)].reset_index(drop=True)
valid_f = valid[~valid["Drug_ID"].isin(dropped_ids)].reset_index(drop=True)
test_f = test[~test["Drug_ID"].isin(dropped_ids)].reset_index(drop=True)
print(f"train={len(train_f)}  valid={len(valid_f)}  test={len(test_f)}")

splits_ames = {
    "dataset": DATASET_NAME, "dev_seed": DEV_SEED,
    "train": {"smiles": train_f["Drug"].tolist(), "y": train_f["Y"].tolist(), "id": train_f["Drug_ID"].tolist()},
    "valid": {"smiles": valid_f["Drug"].tolist(), "y": valid_f["Y"].tolist(), "id": valid_f["Drug_ID"].tolist()},
    "test":  {"smiles": test_f["Drug"].tolist(),  "y": test_f["Y"].tolist(),  "id": test_f["Drug_ID"].tolist()},
}
save_json(splits_ames, AMES_SPLITS_FILE)
print("Saved", AMES_SPLITS_FILE)


Dropping 3 molecules that failed conformer generation.
train=5092  valid=728  test=1455
Saved ../data/splits_ames.json


## Build SELFIES vocab (train-only, same policy as notebook 03)

In [7]:
set_seed(cfg_1d["train"]["seed"])

selfies_by_split = {
    split_name: [smiles_to_selfies(s) for s in splits_ames[split_name]["smiles"]]
    for split_name in ["train", "valid", "test"]
}
vocab = build_vocab([s for s in selfies_by_split["train"] if s is not None])
save_json(vocab, AMES_VOCAB_FILE)
print(f"Vocab size: {len(vocab)}")


Vocab size: 70


## Build all datasets/loaders once, reused by every model below

In [8]:
selfies_datasets = {
    split_name: SelfiesDataset(selfies_by_split[split_name], splits_ames[split_name]["y"], vocab, cfg_1d["model"]["max_len"])
    for split_name in ["train", "valid", "test"]
}
graph_datasets = {
    split_name: [mol_to_graph_data(smi, y) for smi, y in zip(splits_ames[split_name]["smiles"], splits_ames[split_name]["y"])]
    for split_name in ["train", "valid", "test"]
}
conformer_datasets = {
    split_name: ConformerDataset(splits_ames[split_name]["id"], splits_ames[split_name]["y"], AMES_CONFORMERS_DIR)
    for split_name in ["train", "valid", "test"]
}
mm_datasets = {
    split_name: MultimodalDataset(
        ids=splits_ames[split_name]["id"], smiles_list=splits_ames[split_name]["smiles"],
        selfies_list=selfies_by_split[split_name], labels=splits_ames[split_name]["y"],
        vocab=vocab, max_len=cfg_1d["model"]["max_len"], conformers_dir=AMES_CONFORMERS_DIR,
    )
    for split_name in ["train", "valid", "test"]
}
print("all datasets built")


all datasets built


## Train 1D-only

In [9]:
loaders_1d = {
    "train": DataLoader(selfies_datasets["train"], batch_size=cfg_1d["train"]["batch_size"], shuffle=True),
    "valid": DataLoader(selfies_datasets["valid"], batch_size=cfg_1d["train"]["batch_size"], shuffle=False),
    "test":  DataLoader(selfies_datasets["test"],  batch_size=cfg_1d["train"]["batch_size"], shuffle=False),
}
model_1d = Selfies1DModel(vocab_size=len(vocab), pad_id=vocab[PAD], **cfg_1d["model"])

def fwd_1d(model, batch, device):
    return model(batch["input_ids"].to(device))

model_1d, info_1d = train_binary_classifier(model_1d, loaders_1d["train"], loaders_1d["valid"], fwd_1d, cfg_1d, "../experiments/ames_1d_only", device)
test_1d = evaluate(model_1d, loaders_1d["test"], fwd_1d, device)
save_json({"model": "ames_1d_only", **test_1d, **info_1d}, "../experiments/ames_1d_only/metrics.json")
print(f"AMES 1D-only  val_auroc={info_1d['best_val_auroc']:.4f}  test_auroc={test_1d['auroc']:.4f}")


epoch   1  train_loss=0.6253  val_loss=0.5656  val_auroc=0.7857
epoch   2  train_loss=0.5779  val_loss=0.5389  val_auroc=0.8066
epoch   3  train_loss=0.5542  val_loss=0.5191  val_auroc=0.8220
epoch   4  train_loss=0.5217  val_loss=0.5227  val_auroc=0.8172
epoch   5  train_loss=0.5066  val_loss=0.5110  val_auroc=0.8370
epoch   6  train_loss=0.4759  val_loss=0.6104  val_auroc=0.8219
epoch   7  train_loss=0.4545  val_loss=0.5156  val_auroc=0.8197
epoch   8  train_loss=0.4233  val_loss=0.5466  val_auroc=0.7861
epoch   9  train_loss=0.3957  val_loss=0.5279  val_auroc=0.8097
epoch  10  train_loss=0.3586  val_loss=0.5770  val_auroc=0.8050
epoch  11  train_loss=0.3421  val_loss=0.5445  val_auroc=0.8184
epoch  12  train_loss=0.3197  val_loss=0.6309  val_auroc=0.7962
epoch  13  train_loss=0.2894  val_loss=0.6082  val_auroc=0.8054
epoch  14  train_loss=0.2609  val_loss=0.6438  val_auroc=0.7941
epoch  15  train_loss=0.2381  val_loss=0.7274  val_auroc=0.7986
Early stopping at epoch 15 (no val AUROC

## Train 2D-only

In [10]:
loaders_2d = {
    "train": PyGDataLoader(graph_datasets["train"], batch_size=cfg_2d["train"]["batch_size"], shuffle=True),
    "valid": PyGDataLoader(graph_datasets["valid"], batch_size=cfg_2d["train"]["batch_size"], shuffle=False),
    "test":  PyGDataLoader(graph_datasets["test"],  batch_size=cfg_2d["train"]["batch_size"], shuffle=False),
}
model_2d = Graph2DModel(atom_feat_dim=ATOM_FEAT_DIM, bond_feat_dim=BOND_FEAT_DIM, **cfg_2d["model"])

def fwd_2d(model, batch, device):
    batch = batch.to(device)
    return model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)

model_2d, info_2d = train_binary_classifier(model_2d, loaders_2d["train"], loaders_2d["valid"], fwd_2d, cfg_2d, "../experiments/ames_2d_only", device)
test_2d = evaluate(model_2d, loaders_2d["test"], fwd_2d, device)
save_json({"model": "ames_2d_only", **test_2d, **info_2d}, "../experiments/ames_2d_only/metrics.json")
print(f"AMES 2D-only  val_auroc={info_2d['best_val_auroc']:.4f}  test_auroc={test_2d['auroc']:.4f}")


epoch   1  train_loss=0.5778  val_loss=0.5413  val_auroc=0.8052
epoch   2  train_loss=0.5266  val_loss=0.4975  val_auroc=0.8416
epoch   3  train_loss=0.4921  val_loss=0.5129  val_auroc=0.8250
epoch   4  train_loss=0.4859  val_loss=0.4667  val_auroc=0.8492
epoch   5  train_loss=0.4810  val_loss=0.5107  val_auroc=0.8376
epoch   6  train_loss=0.4617  val_loss=0.4509  val_auroc=0.8677
epoch   7  train_loss=0.4602  val_loss=0.4413  val_auroc=0.8730
epoch   8  train_loss=0.4509  val_loss=0.4781  val_auroc=0.8591
epoch   9  train_loss=0.4496  val_loss=0.4477  val_auroc=0.8616
epoch  10  train_loss=0.4393  val_loss=0.4453  val_auroc=0.8648
epoch  11  train_loss=0.4383  val_loss=0.4806  val_auroc=0.8618
epoch  12  train_loss=0.4252  val_loss=0.4383  val_auroc=0.8711
epoch  13  train_loss=0.4291  val_loss=0.4667  val_auroc=0.8582
epoch  14  train_loss=0.4121  val_loss=0.4717  val_auroc=0.8761
epoch  15  train_loss=0.4134  val_loss=0.4673  val_auroc=0.8470
epoch  16  train_loss=0.4041  val_loss=0

## Train 3D-only (learned_attention, locked from hERG's notebook 05 finding)

In [12]:
loaders_3d = {
    "train": DataLoader(conformer_datasets["train"], batch_size=cfg_3d["train"]["batch_size"], shuffle=True, collate_fn=collate_conformers),
    "valid": DataLoader(conformer_datasets["valid"], batch_size=cfg_3d["train"]["batch_size"], shuffle=False, collate_fn=collate_conformers),
    "test":  DataLoader(conformer_datasets["test"],  batch_size=cfg_3d["train"]["batch_size"], shuffle=False, collate_fn=collate_conformers),
}
model_3d = Conformer3DModel(**cfg_3d["model"])

def fwd_3d(model, batch, device):
    return model(
        batch["atom_z"].to(device), batch["atom_pos"].to(device),
        batch["atom_conf_batch"].to(device), batch["conf_mol_batch"].to(device),
        batch["conf_weights"].to(device), num_mols=batch["num_mols"],
    )

model_3d, info_3d = train_binary_classifier(model_3d, loaders_3d["train"], loaders_3d["valid"], fwd_3d, cfg_3d, "../experiments/ames_3d_learned_attention", device)
test_3d = evaluate(model_3d, loaders_3d["test"], fwd_3d, device)
save_json({"model": "ames_3d_learned_attention", **test_3d, **info_3d}, "../experiments/ames_3d_learned_attention/metrics.json")
print(f"AMES 3D-only  val_auroc={info_3d['best_val_auroc']:.4f}  test_auroc={test_3d['auroc']:.4f}")


epoch   1  train_loss=0.7213  val_loss=0.5696  val_auroc=0.7587
epoch   2  train_loss=0.6014  val_loss=0.5511  val_auroc=0.7830
epoch   3  train_loss=0.5822  val_loss=0.5677  val_auroc=0.7627
epoch   4  train_loss=0.5769  val_loss=0.5325  val_auroc=0.7868
epoch   5  train_loss=0.5663  val_loss=0.5395  val_auroc=0.7972
epoch   6  train_loss=0.5536  val_loss=0.5184  val_auroc=0.8047
epoch   7  train_loss=0.5476  val_loss=0.5140  val_auroc=0.8074
epoch   8  train_loss=0.5317  val_loss=0.5404  val_auroc=0.8067
epoch   9  train_loss=0.5242  val_loss=0.5527  val_auroc=0.7723
epoch  10  train_loss=0.5267  val_loss=0.5341  val_auroc=0.8171
epoch  11  train_loss=0.5125  val_loss=0.5380  val_auroc=0.8125
epoch  12  train_loss=0.5103  val_loss=0.5317  val_auroc=0.7988
epoch  13  train_loss=0.4936  val_loss=0.5198  val_auroc=0.8237
epoch  14  train_loss=0.4901  val_loss=0.5435  val_auroc=0.8023
epoch  15  train_loss=0.4763  val_loss=0.5101  val_auroc=0.8307
epoch  16  train_loss=0.4731  val_loss=0

## Train multimodal -- concat and gated

In [13]:
loaders_mm = {
    "train": DataLoader(mm_datasets["train"], batch_size=cfg_fusion["train"]["batch_size"], shuffle=True, collate_fn=multimodal_collate),
    "valid": DataLoader(mm_datasets["valid"], batch_size=cfg_fusion["train"]["batch_size"], shuffle=False, collate_fn=multimodal_collate),
    "test":  DataLoader(mm_datasets["test"],  batch_size=cfg_fusion["train"]["batch_size"], shuffle=False, collate_fn=multimodal_collate),
}

def fwd_mm(model, batch, device):
    graph = batch["graph"].to(device)
    return model(
        batch["input_ids"].to(device),
        graph.x, graph.edge_index, graph.edge_attr, graph.batch,
        batch["conf_atom_z"].to(device), batch["conf_atom_pos"].to(device),
        batch["conf_atom_conf_batch"].to(device), batch["conf_conf_mol_batch"].to(device),
        batch["conf_conf_weights"].to(device), batch["conf_num_mols"],
    )

mm_results = {}
for fusion_type, exp_dir in [("concat", "../experiments/ames_multimodal_concat"), ("gated", "../experiments/ames_multimodal_fusion")]:
    model_mm = MultimodalModel(
        vocab_size=len(vocab), pad_id=vocab[PAD],
        selfies_kwargs=dict(cfg_fusion["encoders"]["selfies"]),
        graph_kwargs={"atom_feat_dim": ATOM_FEAT_DIM, "bond_feat_dim": BOND_FEAT_DIM, **cfg_fusion["encoders"]["graph"]},
        schnet_kwargs=dict(cfg_fusion["encoders"]["schnet"]),
        fusion_type=fusion_type, fusion_dim=cfg_fusion["fusion"]["fusion_dim"],
    )
    model_mm, info_mm = train_binary_classifier(model_mm, loaders_mm["train"], loaders_mm["valid"], fwd_mm, cfg_fusion, exp_dir, device)
    test_mm = evaluate(model_mm, loaders_mm["test"], fwd_mm, device)
    save_json({"model": f"ames_multimodal_{fusion_type}", **test_mm, **info_mm}, f"{exp_dir}/metrics.json")
    mm_results[fusion_type] = (info_mm["best_val_auroc"], test_mm["auroc"])
    print(f"AMES multimodal {fusion_type}  val_auroc={info_mm['best_val_auroc']:.4f}  test_auroc={test_mm['auroc']:.4f}")


epoch   1  train_loss=0.6274  val_loss=0.4952  val_auroc=0.8385
epoch   2  train_loss=0.5245  val_loss=0.4619  val_auroc=0.8604
epoch   3  train_loss=0.5017  val_loss=0.5690  val_auroc=0.8204
epoch   4  train_loss=0.4755  val_loss=0.4676  val_auroc=0.8539
epoch   5  train_loss=0.4613  val_loss=0.4474  val_auroc=0.8662
epoch   6  train_loss=0.4380  val_loss=0.4824  val_auroc=0.8462
epoch   7  train_loss=0.4176  val_loss=0.4446  val_auroc=0.8634
epoch   8  train_loss=0.4043  val_loss=0.4810  val_auroc=0.8540
epoch   9  train_loss=0.3829  val_loss=0.5162  val_auroc=0.8261
epoch  10  train_loss=0.3714  val_loss=0.5483  val_auroc=0.8500
epoch  11  train_loss=0.3463  val_loss=0.5538  val_auroc=0.8391
epoch  12  train_loss=0.3207  val_loss=0.5487  val_auroc=0.8353
epoch  13  train_loss=0.3063  val_loss=0.5277  val_auroc=0.8499
epoch  14  train_loss=0.2872  val_loss=0.6073  val_auroc=0.8065
epoch  15  train_loss=0.2679  val_loss=0.6128  val_auroc=0.8252
Early stopping at epoch 15 (no val AUROC

## Final comparison table + leaderboard reference

In [15]:
ames_table = pd.DataFrame([
    {"model": "1D-only", "val_auroc": info_1d["best_val_auroc"], "test_auroc": test_1d["auroc"]},
    {"model": "2D-only", "val_auroc": info_2d["best_val_auroc"], "test_auroc": test_2d["auroc"]},
    {"model": "3D-only (learned_attention)", "val_auroc": info_3d["best_val_auroc"], "test_auroc": test_3d["auroc"]},
    {"model": "Multimodal (concat)", "val_auroc": mm_results["concat"][0], "test_auroc": mm_results["concat"][1]},
    {"model": "Multimodal (gated)", "val_auroc": mm_results["gated"][0], "test_auroc": mm_results["gated"][1]},
]).sort_values("test_auroc", ascending=False).reset_index(drop=True)

save_json(ames_table.to_dict(orient="records"), "../experiments/ames_final_comparison.json")
ames_table


,model,val_auroc,test_auroc
0,2D-only,0.881671,0.836379
1,3D-only (learned_attention),0.849672,0.804143
2,Multimodal (concat),0.866203,0.786212
3,1D-only,0.837026,0.723216
4,Multimodal (gated),0.834309,0.713868


## AMES leaderboard reference (official, fetched from tdcommons.ai)

Mean +/- std across TDC's 5 official seeds; this is single-seed, so directional only:

| Rank | Model | AUROC |
|---|---|---|
| 1 | ZairaChem | 0.871 +/- 0.002 |
| 3 | MapLight + GNN | 0.869 +/- 0.002 |
| 6 | Chemprop-RDKit | 0.850 +/- 0.004 |
| 15 | GCN (GNN baseline) | 0.818 +/- 0.010 |
| 16 | AttentiveFP (GNN baseline) | 0.814 +/- 0.008 |

